In [1]:
from dotenv import load_dotenv
from src.db.chroma_db import ChromaDb
from src.models.openai_provider import OpenAILLMProvider, OpenAIEmbeddingProvider
from src.prompts import timeframe_detection_system, timeframe_detection_user
from src.schemas.schemas import TimeframeDetection, InputLanguage
from src.services.nkod_data_processor import NkodDataProcessor
from src.db.graph_db import GraphDb
from src.db.sq_lite import SqLite
from datetime import date
from src.services.language_detector import LanguageDetector
from src.services.nkod_query_matcher import NkodQueryMatcher
from src.services.timeframe_detector import TimeframeDetector
from src.evaluators.nkod_query_matcher_evaluator import NkodQueryMatcherEvaluator
from src.services.nkod_query_matcher_reranker import NkodQueryMatcherReranker
from src.services.nkod_rag import NkodRAG


load_dotenv()

2025-12-01 01:24:31.604005710 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"
/home/lamossta/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## Downloading and creating SQL tables for the NKOD metadata


In [3]:
import pandas as pd
from src.services.nkod_data_processor import NkodDataProcessor
from src.utils import dir_name_from_uri

metadata_df = pd.read_csv(NkodDataProcessor("nkod").ofn_metadata_csv_path)
print(metadata_df.shape)
#metadata_df = metadata_df[metadata_df['dataset_uri'].apply(dir_name_from_uri) != "999788533"]
#metadata_df.to_csv(NkodDataProcessor("nkod").ofn_metadata_csv_path, index=False)

(511, 12)


In [ ]:
nkod_data_processor = NkodDataProcessor("nkod")
graph_db = GraphDb(nkod_data_processor.catalog_name)
sq_lite = SqLite(nkod_data_processor.metadata_sql_path)

#nkod_data_processor.create_dataset_publisher_csv(graph_db)
nkod_data_processor.download_catalog_metadata()
nkod_data_processor.download_catalog_distributions()
nkod_data_processor.download_catalog_datasets()
nkod_data_processor.create_metadata_csv(graph_db)
nkod_data_processor.create_themes_csv(graph_db)
nkod_data_processor.create_metadata_sql(sq_lite)
nkod_data_processor.create_themes_sql(sq_lite)

In [8]:
from src.pipelines.nkod_query_matcher_pipeline import NkodQueryMatcherPipeline
from src.schemas.nkod_query_matcher_request import NkodQueryMatcherRequest
from src.services.entity_generator import EntityGenerator
from src.services.nkod_data_processor import NkodDataProcessor
from src.models.openai_provider import OpenAILLMProvider, OpenAIEmbeddingProvider
from src.db.chroma_db import ChromaDb
from src.services.nkod_query_matcher import NkodQueryMatcher

nkod_data_processor = NkodDataProcessor("nkod")
openai_embeddings = OpenAIEmbeddingProvider(model_name="text-embedding-3-large", dimensions=None)
chroma_db = ChromaDb(nkod_data_processor.vectordb_path)
query = "Jaký druh stromu se v posledním roce procentuálně nejvíce obnovoval v Libereckém kraji?"
nkod_query_matcher = NkodQueryMatcher(query)
entity_generator = EntityGenerator()
#print(nkod_query_matcher.get_matching_entitities_keywords(30, chroma_db, nkod_data_processor, "cs", openai_embeddings, entity_generator))


rq = NkodQueryMatcherRequest(
    query=query,
    llm_provider="openai",
    language="cs",
    model_name="gpt-4.1"
)

matching_pipeline = NkodQueryMatcherPipeline().run(rq)

started
Elapsed time: 5.3083 seconds
Elapsed time: 14.0425 seconds
Elapsed time: 15.1274 seconds
Elapsed time: 16.5070 seconds
Elapsed time: 16.9917 seconds
Elapsed time: 21.7246 seconds
Elapsed time: 26.3085 seconds
new
dict_values([[{'dataset_uri': 'https://data.gov.cz/zdroj/datové-sady/70890749/990091919', 'description_cs': 'Tato datová sada obsahuje data z úředních desek dle Otevřené formální normy Úřední desky (https://ofn.gov.cz/úřední-desky/2021-07-20/).', 'description_en': 'This dataset contains information from official bulletin boards according to the formal open standard.', 'has_rdf_distribution': 1, 'keywords_cs': 'úřední deska', 'keywords_en': 'bulletin board', 'matched_substring': 'Úřední deska', 'publisher_cs': 'Kraj Vysočina', 'publisher_en': 'None', 'themes': 'GOVE', 'title_cs': 'Úřední deska - Kraj Vysočina', 'title_en': 'Official bulletin board - Vysocina Region', 'score': 0.5273013114929199, 'doc': 'úřední deska - kraj vysočina', 'matched_on': 'Entity', 'relevance_s

In [5]:
import pandas as pd

data = [
    {"name": "Alice", "age": 30, "city": "Sydney"},
    {"name": "Bob", "age": 25, "city": "Melbourne"},
    {"name": "Charlie", "age": 35, "city": "Brisbane"}
]

df = pd.DataFrame(data)
print(df)

      name  age       city
0    Alice   30     Sydney
1      Bob   25  Melbourne
2  Charlie   35   Brisbane


## Indexing the keywords, titles and descriptions from the NKOD metadata

In [3]:
openai_embeddings = OpenAIEmbeddingProvider(model_name="text-embedding-3-large", dimensions=1536)
chroma_db = ChromaDb(nkod_data_processor.vectordb_path)

nkod_data_processor.index_catalog_themes(sq_lite, openai_embeddings, chroma_db)
nkod_data_processor.index_catalog_metadata(sq_lite, openai_embeddings, chroma_db, verbose=True)
print(chroma_db.list_collections())

Created or loaded collection 'nkod_themes_labels_cs'
Created or loaded collection 'nkod_themes_labels_en'
Created or loaded collection 'nkod_themes_definitions_cs'
Created or loaded collection 'nkod_themes_definitions_en'
Created or loaded collection 'nkod_keywords_cs'


Adding document batches of keywords_cs: 100%|██████████| 1/1 [00:12<00:00, 12.53s/it]


Mean number of cs keywords: 2.2902418682235197
Created or loaded collection 'nkod_keywords_en'


Adding document batches of keywords_en: 100%|██████████| 1/1 [00:06<00:00,  6.43s/it]


Mean number of en keywords: 0.8457047539616347
Created or loaded collection 'nkod_descriptions_cs'


...

Mean number of en descriptions that are not None: 1.0
Created or loaded collection 'nkod_titles_cs'


Adding document batches of title_cs: 100%|██████████| 1/1 [00:07<00:00,  7.93s/it]


Mean number of cs titles that are not None: 1.0
Created or loaded collection 'nkod_titles_en'


Adding document batches of title_en: 100%|██████████| 1/1 [00:03<00:00,  3.43s/it]

Mean number of en titles that are not None: 1.0
['nkod_keywords_cs', 'nkod_themes_definitions_en', 'nkod_titles_cs', 'nkod_themes_labels_en', 'nkod_themes_definitions_cs', 'nkod_titles_en', 'nkod_themes_labels_cs', 'nkod_keywords_en', 'nkod_descriptions_cs', 'nkod_descriptions_en']


## Language detection, Timeframe detection and Query matching on OFN dataset

In [ ]:
model_name ="gpt-5"
openai_llm = OpenAILLMProvider(
    model_name=model_name,
    temperature=1.0
)
nkod_query_evaluator = NkodQueryMatcherEvaluator()
nkod_query_reranker = NkodQueryMatcherReranker()

k = 30
evaluation_dataset = "ofn_dataset_ofn_new.jsonl"
nkod_query_evaluator.evaluate_on_ofn_dataset(k, evaluation_dataset, chroma_db, nkod_data_processor, "cs", openai_embeddings, nkod_query_reranker, openai_llm)

## Language detection, Timeframe detection and Query matching on LLM dataset

In [ ]:
from rdflib import Graph
g = Graph()
g.parse("https://data.mff.cuni.cz/soubory/číselníky/organizační-struktura.ofn.jsonld", format="json-ld")
print(list(g.query("""
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>\n
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>\n
SELECT DISTINCT ?rel ?com\n
        WHERE { \n
        ?rel a/rdfs:subPropertyOf* rdf:Property . \n
        OPTIONAL { ?rel rdfs:comment ?com } \n
        }
""")))

In [33]:
from shaclgen.shaclgen import data_graph
from rdflib import Graph

source_graph = Graph()
source_graph.parse("https://data.mff.cuni.cz/soubory/čoi/coi.trig", format="trig")

extraction_graph = data_graph(source_graph)
shacl_graph = extraction_graph.gen_graph()
print(shacl_graph)

2025-10-23 15:51:52.087 | INFO     | shaclgen.shaclgen:gen_graph:119 - Start Extraction of the Data Graph
2025-10-23 15:51:52.122 | INFO     | shaclgen.shaclgen:gen_graph:120 - Classes …
2025-10-23 15:51:52.409 | INFO     | shaclgen.shaclgen:gen_graph:122 - Properties …
2025-10-23 15:51:52.419 | INFO     | shaclgen.shaclgen:gen_graph:124 - Write resulting SHACL Graph …


[a rdfg:Graph;rdflib:storage [a rdflib:Store;rdfs:label 'Memory']].


In [46]:
print(shacl_graph.serialize(format='trig'))
shacl_graph.print("trig")
